In [1]:
import pandas as pd
import numpy as np
import re
import ipaddress
from urllib.parse import urlparse

In [2]:
def extract_basic_url_features(url):
    parsed = urlparse(url)

    # Make sure URL has a scheme
    if not parsed.netloc:
        parsed = urlparse("http://" + url)

    domain = parsed.netloc.split(":")[0]
    path = parsed.path
    query = parsed.query

    features = {}

    # Basic URL information
    features["URLLength"] = len(url)
    features["DomainLength"] = len(domain)

    # Is domain an IP address?
    try:
        ipaddress.ip_address(domain)
        features["IsDomainIP"] = 1
    except ValueError:
        features["IsDomainIP"] = 0

    # HTTPS
    features["IsHTTPS"] = 1 if parsed.scheme.lower() == "https" else 0

    # Domain / subdomain
    domain_parts = domain.split(".")
    features["NoOfSubDomain"] = max(len(domain_parts) - 2, 0)

    # TLD
    if len(domain_parts) >= 2:
        tld = domain_parts[-1]
    else:
        tld = ""

    features["TLDLength"] = len(tld)

    # Character counts
    features["NoOfLettersInURL"] = sum(c.isalpha() for c in url)
    features["NoOfDegitsInURL"] = sum(c.isdigit() for c in url)

    features["NoOfEqualsInURL"] = url.count("=")
    features["NoOfQMarkInURL"] = url.count("?")
    features["NoOfAmpersandInURL"] = url.count("&")

    # Special characters
    special_chars = re.findall(r'[^a-zA-Z0-9]', url)

    features["NoOfOtherSpecialCharsInURL"] = len(special_chars)

    # Ratios
    url_length = max(len(url), 1)

    features["LetterRatioInURL"] = (
        features["NoOfLettersInURL"] / url_length
    )

    features["DegitRatioInURL"] = (
        features["NoOfDegitsInURL"] / url_length
    )

    features["SpacialCharRatioInURL"] = (
        features["NoOfOtherSpecialCharsInURL"] / url_length
    )

    return features

In [3]:
test_url = "https://www.example.com/login?id=123"

features = extract_basic_url_features(test_url)

features

{'URLLength': 36,
 'DomainLength': 15,
 'IsDomainIP': 0,
 'IsHTTPS': 1,
 'NoOfSubDomain': 1,
 'TLDLength': 3,
 'NoOfLettersInURL': 25,
 'NoOfDegitsInURL': 3,
 'NoOfEqualsInURL': 1,
 'NoOfQMarkInURL': 1,
 'NoOfAmpersandInURL': 0,
 'NoOfOtherSpecialCharsInURL': 8,
 'LetterRatioInURL': 0.6944444444444444,
 'DegitRatioInURL': 0.08333333333333333,
 'SpacialCharRatioInURL': 0.2222222222222222}

In [4]:
df = pd.read_csv("../datasets/phishing_urls/Phishing_url.csv")

In [5]:
sample_url = df.iloc[0]["URL"]

print(sample_url)

our_features = extract_basic_url_features(sample_url)

our_features

https://www.southbankmosaics.com


{'URLLength': 32,
 'DomainLength': 24,
 'IsDomainIP': 0,
 'IsHTTPS': 1,
 'NoOfSubDomain': 1,
 'TLDLength': 3,
 'NoOfLettersInURL': 27,
 'NoOfDegitsInURL': 0,
 'NoOfEqualsInURL': 0,
 'NoOfQMarkInURL': 0,
 'NoOfAmpersandInURL': 0,
 'NoOfOtherSpecialCharsInURL': 5,
 'LetterRatioInURL': 0.84375,
 'DegitRatioInURL': 0.0,
 'SpacialCharRatioInURL': 0.15625}

In [6]:
columns_to_compare = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLDLength",
    "NoOfSubDomain",
    "NoOfLettersInURL",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "IsHTTPS"
]

df.iloc[0][columns_to_compare]

URLLength             31
DomainLength          24
IsDomainIP             0
TLDLength              3
NoOfSubDomain          1
NoOfLettersInURL      18
NoOfDegitsInURL        0
NoOfEqualsInURL        0
NoOfQMarkInURL         0
NoOfAmpersandInURL     0
IsHTTPS                1
Name: 0, dtype: object

In [7]:
url = df.iloc[0]["URL"]

print("URL:", repr(url))
print("Python len:", len(url))

parsed = urlparse(url)

print("\nScheme:", repr(parsed.scheme))
print("Netloc:", repr(parsed.netloc))
print("Path:", repr(parsed.path))
print("Query:", repr(parsed.query))
print("Fragment:", repr(parsed.fragment))

print("\nLetters in complete URL:",
      sum(c.isalpha() for c in url))

print("Letters in domain:",
      sum(c.isalpha() for c in parsed.netloc))

print("Letters in path:",
      sum(c.isalpha() for c in parsed.path))

print("\nCharacters:")
print(list(enumerate(url)))

URL: 'https://www.southbankmosaics.com'
Python len: 32

Scheme: 'https'
Netloc: 'www.southbankmosaics.com'
Path: ''
Query: ''
Fragment: ''

Letters in complete URL: 27
Letters in domain: 22
Letters in path: 0

Characters:
[(0, 'h'), (1, 't'), (2, 't'), (3, 'p'), (4, 's'), (5, ':'), (6, '/'), (7, '/'), (8, 'w'), (9, 'w'), (10, 'w'), (11, '.'), (12, 's'), (13, 'o'), (14, 'u'), (15, 't'), (16, 'h'), (17, 'b'), (18, 'a'), (19, 'n'), (20, 'k'), (21, 'm'), (22, 'o'), (23, 's'), (24, 'a'), (25, 'i'), (26, 'c'), (27, 's'), (28, '.'), (29, 'c'), (30, 'o'), (31, 'm')]


In [8]:
test_url = "https://www.southbankmosaics.com"

row = df[df["URL"] == test_url]

if len(row) > 0:
    print(row[[
        "URL",
        "URLLength",
        "Domain",
        "DomainLength",
        "IsDomainIP",
        "TLD",
        "TLDLength",
        "NoOfSubDomain",
        "NoOfLettersInURL",
        "NoOfDegitsInURL",
        "NoOfEqualsInURL",
        "NoOfQMarkInURL",
        "NoOfAmpersandInURL",
        "NoOfOtherSpecialCharsInURL",
        "IsHTTPS"
    ]].T)
else:
    print("URL not found")

                                                           0
URL                         https://www.southbankmosaics.com
URLLength                                                 31
Domain                              www.southbankmosaics.com
DomainLength                                              24
IsDomainIP                                                 0
TLD                                                      com
TLDLength                                                  3
NoOfSubDomain                                              1
NoOfLettersInURL                                          18
NoOfDegitsInURL                                            0
NoOfEqualsInURL                                            0
NoOfQMarkInURL                                             0
NoOfAmpersandInURL                                         0
NoOfOtherSpecialCharsInURL                                 1
IsHTTPS                                                    1


In [9]:
from urllib.parse import urlparse

url = "https://www.southbankmosaics.com"

parsed = urlparse(url)

print("Python len:", len(url))
print("Scheme:", parsed.scheme)
print("Netloc:", parsed.netloc)
print("Hostname:", parsed.hostname)
print("Path:", parsed.path)
print("Query:", parsed.query)
print("Fragment:", parsed.fragment)

Python len: 32
Scheme: https
Netloc: www.southbankmosaics.com
Hostname: www.southbankmosaics.com
Path: 
Query: 
Fragment: 


In [10]:
# Select 10 random URLs from the dataset
sample_urls = df.sample(10, random_state=42)

cols = [
    'URL',
    'URLLength',
    'DomainLength',
    'IsDomainIP',
    'TLD',
    'TLDLength',
    'NoOfSubDomain',
    'NoOfLettersInURL',
    'NoOfDegitsInURL',
    'NoOfEqualsInURL',
    'NoOfQMarkInURL',
    'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL',
    'IsHTTPS'
]

display(sample_urls[cols])

,URL,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,NoOfLettersInURL,NoOfDegitsInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,IsHTTPS
136221,https://www.northcm.ac.th,24,17,0,th,2,2,10,0,0,0,0,2,1
56609,https://unitedmartialartscenters.com/at0/mygov...,59,28,0,com,3,0,45,1,0,0,0,5,1
46393,https://email.mail1.onesignal.os.tc/c/ejwcz02u...,385,27,0,tc,2,3,302,51,0,0,0,24,1
129746,http://uqr.to/1il1z,19,6,0,to,2,0,8,2,0,0,0,2,0
131464,https://www.woolworthsrewards.com.au,35,28,0,au,2,2,21,0,0,0,0,2,1
233369,https://www.kubiena-kochblog.com,31,24,0,com,3,1,17,0,0,0,0,2,1
167698,https://couchjumper.com/covantage/index.html,44,15,0,com,3,0,32,0,0,0,0,4,1
157482,http://www.rightnowscope.com,27,21,0,com,3,1,15,0,0,0,0,1,0
64913,http://u1965047.plsk.regruhosting.ru/47/,40,29,0,ru,2,2,19,9,0,0,0,5,0
156754,http://www.unblock-me.org,24,18,0,org,3,1,11,0,0,0,0,2,0


In [11]:
for url in sample_urls['URL']:
    print("=" * 80)
    print(url)
    print("Python len:", len(url))

https://www.northcm.ac.th
Python len: 25
https://unitedmartialartscenters.com/at0/mygov/personal.html
Python len: 60
https://email.mail1.onesignal.os.tc/c/ejwcz02untamqohvwmwoczwfbgw6uduohnspqdycugml7r66b3ikm_n4vn8euqvy0apkcfu8eicckzyerqtqfuujn-uwkk7aztt8kt6xqthp-8zefetokzpt6kio3dr7du0vo7x52ipwpli8sk4erixqimbifemtuqimy2ljpdaro7vwuff3mm9th-n-t-hhhk8jx2e_ebel9efrpvh-l9k_jnx994qv2u32-erjaifwopr1wywwoqejkeqqyktbdshcglptnaswjp7vk__z89kvwzbzftigbbjhkeoi4lionzk1wudnu74oo5e__ap_nl8tuwrvftlyzp8faad__-czz7m
Python len: 385
http://uqr.to/1il1z
Python len: 19
https://www.woolworthsrewards.com.au
Python len: 36
https://www.kubiena-kochblog.com
Python len: 32
https://couchjumper.com/covantage/index.html
Python len: 44
http://www.rightnowscope.com
Python len: 28
http://u1965047.plsk.regruhosting.ru/47/
Python len: 40
http://www.unblock-me.org
Python len: 25


In [34]:
%%writefile ../src/phishing/url_features.py
import re
from urllib.parse import urlparse
import tldextract

def extract_18_url_features(raw_url: str) -> dict:
    """
    Extracts the 18 URL-derived lexical features matching PhiUSIIL dataset rules.
    """
    # 1. Normalize trailing slash for length compatibility
    clean_url = raw_url.rstrip('/')
    url_length = len(clean_url)
    
    # 2. Parse URL components
    parsed = urlparse(clean_url)
    ext = tldextract.extract(clean_url)
    
    domain_str = parsed.netloc if parsed.netloc else parsed.path.split('/')[0]
    domain_length = len(domain_str)
    
    # 3. IP check
    is_domain_ip = 1 if re.match(r'^\d{1,3}(\.\d{1,3}){3}$', domain_str) else 0
    
    # 4. TLD and Subdomain features
    tld = ext.suffix
    tld_length = len(tld)
    
    subdomain = ext.subdomain
    # Count subdomains excluding 'www'
    subdomains_list = [s for s in subdomain.split('.') if s and s != 'www']
    no_of_subdomain = len(subdomains_list)
    
    # 5. Domain-body letter count (matching PhiUSIIL SLD logic)
    sld_domain = ext.domain
    no_of_letters = sum(c.isalpha() for c in sld_domain)
    letter_ratio = no_of_letters / url_length if url_length > 0 else 0.0
    
    # 6. Digits in entire URL string
    no_of_digits = sum(c.isdigit() for c in clean_url)
    digit_ratio = no_of_digits / url_length if url_length > 0 else 0.0
    
    # 7. Character counts
    no_of_equals = clean_url.count('=')
    no_of_qmark = clean_url.count('?')
    no_of_ampersand = clean_url.count('&')
    
    # Standard special characters excluded from alphanumeric check
    alnum_or_standard_specials = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789=&#?.-:/_")
    no_of_other_specials = sum(1 for c in clean_url if c not in alnum_or_standard_specials)
    special_char_ratio = no_of_other_specials / url_length if url_length > 0 else 0.0
    
    # 8. HTTPS check
    is_https = 1 if clean_url.lower().startswith('https://') else 0
    
    # 9. Obfuscation features
    has_obfuscation = 1 if '%' in clean_url or '@' in clean_url else 0
    no_of_obfuscated_char = clean_url.count('%') + clean_url.count('@')
    obfuscation_ratio = no_of_obfuscated_char / url_length if url_length > 0 else 0.0
    
    return {
        'URLLength': url_length,
        'DomainLength': domain_length,
        'IsDomainIP': is_domain_ip,
        'TLDLength': tld_length,
        'NoOfSubDomain': no_of_subdomain,
        'HasObfuscation': has_obfuscation,
        'NoOfObfuscatedChar': no_of_obfuscated_char,
        'ObfuscationRatio': obfuscation_ratio,
        'NoOfLettersInURL': no_of_letters,
        'LetterRatioInURL': letter_ratio,
        'NoOfDegitsInURL': no_of_digits,
        'DegitRatioInURL': digit_ratio,
        'NoOfEqualsInURL': no_of_equals,
        'NoOfQMarkInURL': no_of_qmark,
        'NoOfAmpersandInURL': no_of_ampersand,
        'NoOfOtherSpecialCharsInURL': no_of_other_specials,
        'SpacialCharRatioInURL': special_char_ratio,
        'IsHTTPS': is_https
    }

Writing ../src/phishing/url_features.py


In [36]:
!pip install tldextract


   ---------------------------------------- 2/2 [tldextract]



In [37]:
import sys
import os

# Add src folder to path
sys.path.append(os.path.abspath('../src'))

from phishing.url_features import extract_18_url_features

correct = 0
total = 0

for i in range(100):
    url = df.iloc[i]["URL"]
    ours = extract_18_url_features(url)

    if ours["DomainLength"] == df.iloc[i]["DomainLength"]:
        correct += 1

    total += 1

print("DomainLength accuracy:", correct / total)

DomainLength accuracy: 1.0


In [38]:
print(df.shape)
print(df.columns.tolist())

(235795, 55)
['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier

# 1. Extract all 18 features for each URL
print("Extracting 18 lexical features from dataset URLs...")
features_list = [extract_18_url_features(url) for url in df['URL']]
X = pd.DataFrame(features_list)

# 2. Target variable (check whether dataset column is named 'label' or 'label')
y = df['label'] if 'label' in df.columns else df['label']

# 3. Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 4. Train Decision Tree Baseline Model
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 5. Predict & Evaluate
preds = model.predict(X_test)
print(f"18-Feature Decision Tree Accuracy: {accuracy_score(y_test, preds):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, preds))

Extracting 18 lexical features from dataset URLs...
18-Feature Decision Tree Accuracy: 0.9661

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.93      0.96     20189
           1       0.95      0.99      0.97     26970

    accuracy                           0.97     47159
   macro avg       0.97      0.96      0.97     47159
weighted avg       0.97      0.97      0.97     47159



In [41]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd

# Define the 4 target classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear SVM": SGDClassifier(loss='hinge', max_iter=1000, random_state=42, n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
}

results = []

print("Training & Comparing 4 Models on 18 Lexical Features...\n")

for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

# Render performance comparison as a Clean Summary DataFrame
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
results_df.reset_index(drop=True, inplace=True)
print(results_df.to_string(index=False))

Training & Comparing 4 Models on 18 Lexical Features...

              Model  Accuracy  Precision  Recall  F1-Score
            XGBoost    0.9889     0.9879  0.9928    0.9903
      Random Forest    0.9810     0.9758  0.9914    0.9835
Logistic Regression    0.9755     0.9746  0.9827    0.9786
         Linear SVM    0.9725     0.9811  0.9707    0.9759


In [42]:
import joblib
import os

# 1. Train top-performing XGBoost model on the full feature dataset (X, y)
print("Training final XGBoost model on full dataset...")
best_model = XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1, 
    random_state=42, 
    n_jobs=-1
)
best_model.fit(X, y)

# 2. Define artifact directory path
export_dir = '../models/phishing'
os.makedirs(export_dir, exist_ok=True)
model_path = os.path.join(export_dir, 'xgboost_phishing_model.pkl')

# 3. Create deployment payload containing model and ordered feature list
artifact = {
    'model': best_model,
    'feature_names': list(X.columns)
}

# 4. Save model artifact
joblib.dump(artifact, model_path)
print(f"Successfully saved trained model artifact to: {model_path}")

Training final XGBoost model on full dataset...
Successfully saved trained model artifact to: ../models/phishing\xgboost_phishing_model.pkl


In [50]:
import joblib
import pandas as pd
import sys
import os
from urllib.parse import urlparse

# 1. Load model artifact
artifact_path = '../models/phishing/xgboost_phishing_model.pkl'
artifact = joblib.load(artifact_path)
model = artifact['model']
feature_names = artifact['feature_names']

# 2. Known Safe Root Domains Whitelist
SAFE_DOMAINS = {"google.com", "github.com", "microsoft.com", "apple.com", "amazon.com", "wikipedia.org"}

def predict_url_safety(raw_url: str):
    # Check domain whitelist
    domain = urlparse(raw_url).netloc.lower()
    if domain.startswith("www."):
        domain = domain[4:]
        
    if domain in SAFE_DOMAINS:
        return {"verdict": "Legitimate", "confidence": 100.0, "reason": "Whitelisted Domain"}
    
    # Extract features & predict via XGBoost
    features_dict = extract_18_url_features(raw_url)
    input_df = pd.DataFrame([features_dict])[feature_names]
    
    prediction = model.predict(input_df)[0]
    probs = model.predict_proba(input_df)[0]
    
    label_map = {0: "Phishing", 1: "Legitimate"}
    return {
        "verdict": label_map[prediction],
        "confidence": probs[prediction] * 100,
        "reason": "XGBoost Model Prediction"
    }

# Test GitHub URL again
res = predict_url_safety("https://github.com/torvalds/linux")
print(f"Verdict: {res['verdict']} | Confidence: {res['confidence']:.2f}% | Source: {res['reason']}")

Verdict: Legitimate | Confidence: 100.00% | Source: Whitelisted Domain


classifier.py TEST

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

from phishing.classifier import PhishingClassifier


MODEL_PATH = "../models/phishing/xgboost_phishing_model.pkl"

classifier = PhishingClassifier(MODEL_PATH)


test_urls = [
    "https://google.com",
    "https://github.com",
    "https://example.com",
    "http://uqr.to/1il1z"
]

for url in test_urls:

    result = classifier.predict(url)

    print("\nURL:", url)
    print("Result:", result)


URL: https://google.com
Result: {'verdict': 'Legitimate', 'confidence': 100.0, 'reason': 'Whitelisted Domain'}

URL: https://github.com
Result: {'verdict': 'Legitimate', 'confidence': 100.0, 'reason': 'Whitelisted Domain'}

URL: https://example.com
Result: {'verdict': 'Phishing', 'confidence': 99.34636688232422, 'reason': 'XGBoost Model Prediction'}

URL: http://uqr.to/1il1z
Result: {'verdict': 'Phishing', 'confidence': 99.99324035644531, 'reason': 'XGBoost Model Prediction'}
